<a href="https://colab.research.google.com/github/dhiru69-tech/ML-Techniques-Lab-Project/blob/main/notebooks/day05_encoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 5: Encoding Categorical Features
**Course:** BO CDA 205, Machine Learning Techniques
**Dataset:** German Credit (1000 customers, 21 columns)
**Syllabus link:** Foundation (preparing data for every algorithm that follows)

> **Goal of today:** Turn the 13 text columns and the target into numbers without introducing a fake order.

In [1]:
import pandas as pd
from sklearn.datasets import fetch_openml

# Load the German Credit dataset as a table
data = fetch_openml(name="credit-g", version=1, as_frame=True)
df = data.frame
print(df.shape)   # should print (1000, 21)

# Separate column names by type
num_cols = df.select_dtypes(include='number').columns.tolist()
cat_cols = df.select_dtypes(exclude='number').columns.tolist()
cat_cols.remove('class')      # 'class' is the target, handled separately
print("Numeric:", len(num_cols), "| Categorical features:", len(cat_cols))

(1000, 21)
Numeric: 7 | Categorical features: 13


## 2. Cardinality of Each Categorical Feature

**Cardinality** = number of different categories in a column.

> **Why it matters:** one-hot encoding creates one new column per category, so high-cardinality columns make the table wide.

In [2]:
# nunique() counts the different values in each column
# sort_values() puts the biggest cardinality first
print(df[cat_cols].nunique().sort_values(ascending=False))

# Total number of columns we expect after one-hot encoding (no column dropped)
print("Expected one-hot columns:", df[cat_cols].nunique().sum())

purpose                10
savings_status          5
credit_history          5
employment              5
checking_status         4
personal_status         4
property_magnitude      4
job                     4
other_parties           3
other_payment_plans     3
housing                 3
own_telephone           2
foreign_worker          2
dtype: int64
Expected one-hot columns: 54


## 3. Looking at the Actual Categories

Before choosing an encoding we must see the labels. The question to ask for each column: **is there a natural order?**

In [3]:
# Print the categories of a few columns to decide nominal vs ordinal
for col in ['checking_status', 'savings_status', 'employment', 'purpose', 'housing']:
    print(col, "->", list(df[col].cat.categories))

checking_status -> ['0<=X<200', '<0', '>=200', 'no checking']
savings_status -> ['100<=X<500', '500<=X<1000', '<100', '>=1000', 'no known savings']
employment -> ['1<=X<4', '4<=X<7', '<1', '>=7', 'unemployed']
purpose -> ['domestic appliance', 'new car', 'used car', 'business', 'education', 'furniture/equipment', 'other', 'radio/tv', 'repairs', 'retraining']
housing -> ['for free', 'own', 'rent']


## 4. Encoding the Target (Label Encoding)

The target has two values, so a simple mapping works: **bad = 1, good = 0**.

> **Why bad = 1?** The bank cares about catching bad customers, so the "positive" class that we try to detect is `bad`. Metrics like recall and precision (Day 8) refer to this positive class.

In [4]:
# Map text labels to numbers using a dictionary
y = df['class'].map({'good': 0, 'bad': 1})

# Check the result: the counts must match what we saw on Day 2 (700 and 300)
print(y.value_counts())

class
0    700
1    300
Name: count, dtype: int64


## 5. Ordinal Encoding (when we tell Python the order)

We use it only when a real order exists. We must **list the order ourselves**, otherwise Python sorts alphabetically, which is usually wrong.

> **Demo only:** `employment` roughly goes from short to long service. We do this to learn the tool. For the final model we still prefer one-hot for this column.

In [5]:
from sklearn.preprocessing import OrdinalEncoder

# First look at the exact spelling of the categories
print(list(df['employment'].cat.categories))

['1<=X<4', '4<=X<7', '<1', '>=7', 'unemployed']


In [6]:
# Write the order from LOWEST to HIGHEST, using the exact spelling printed above
order = ['unemployed', '<1', '1<=X<4', '4<=X<7', '>=7']

enc = OrdinalEncoder(categories=[order])

# fit_transform expects a 2D table, so we use double brackets [['employment']]
emp_encoded = enc.fit_transform(df[['employment']])

# Compare original text with the number
demo = pd.DataFrame({'text': df['employment'], 'number': emp_encoded.ravel()})
print(demo.drop_duplicates().sort_values('number'))

          text  number
9   unemployed     0.0
10          <1     1.0
1       1<=X<4     2.0
2       4<=X<7     3.0
0          >=7     4.0


## 6. One-Hot Encoding with pandas (`get_dummies`)

Each category becomes its own 0/1 column.

**Dummy variable trap:** with all columns kept, one column can be predicted from the rest. `drop_first=True` removes one column per feature.

In [7]:
# Features only: drop the target column
X = df.drop(columns=['class'])

# One-hot encode all categorical columns, keep every category column
# dtype=int gives 0/1 instead of True/False
X_full = pd.get_dummies(X, columns=cat_cols, dtype=int)
print("With all dummy columns:", X_full.shape)

# Same, but drop one column per feature to avoid redundancy
X_drop = pd.get_dummies(X, columns=cat_cols, drop_first=True, dtype=int)
print("With drop_first=True:", X_drop.shape)

With all dummy columns: (1000, 61)
With drop_first=True: (1000, 48)


In [8]:
# Look at a few of the new columns
print(X_drop.columns.tolist()[:15])
X_drop.head()

['duration', 'credit_amount', 'installment_commitment', 'residence_since', 'age', 'existing_credits', 'num_dependents', 'checking_status_<0', 'checking_status_>=200', 'checking_status_no checking', 'credit_history_critical/other existing credit', 'credit_history_delayed previously', 'credit_history_existing paid', 'credit_history_no credits/all paid', 'purpose_new car']


,duration,credit_amount,installment_commitment,residence_since,age,existing_credits,num_dependents,checking_status_<0,checking_status_>=200,checking_status_no checking,...,property_magnitude_car,other_payment_plans_none,other_payment_plans_stores,housing_own,housing_rent,job_unemp/unskilled non res,job_unskilled resident,job_skilled,own_telephone_yes,foreign_worker_yes
0,6,1169,4,4,67,2,1,1,0,0,...,0,1,0,1,0,0,0,1,1,1
1,48,5951,2,2,22,1,1,0,0,0,...,0,1,0,1,0,0,0,1,0,1
2,12,2096,2,3,49,1,2,0,0,1,...,0,1,0,1,0,0,1,0,0,1
3,42,7882,2,4,45,1,2,1,0,0,...,0,1,0,0,0,0,0,1,0,1
4,24,4870,3,4,53,2,2,1,0,0,...,0,1,0,0,0,0,0,1,0,1


## 7. The Proper Way: `OneHotEncoder`

`get_dummies` is quick for exploring, but it does not remember what it learned. `OneHotEncoder` is a **fit / transform** tool, like the scaler on Day 6:

- **fit** on training data: learns which categories exist
- **transform** on any data: applies the same columns

`handle_unknown='ignore'` means a category never seen in training becomes all zeros instead of crashing.

In [9]:
from sklearn.preprocessing import OneHotEncoder

# sparse_output=False gives a normal table instead of a sparse matrix
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Learn the categories and transform in one step
encoded = ohe.fit_transform(df[cat_cols])
print("Encoded shape:", encoded.shape)

# Names of the new columns, e.g. housing_own
new_names = ohe.get_feature_names_out(cat_cols)
print(list(new_names[:10]))

# Put it back into a readable table
encoded_df = pd.DataFrame(encoded, columns=new_names, index=df.index)
encoded_df.head()

Encoded shape: (1000, 54)
['checking_status_0<=X<200', 'checking_status_<0', 'checking_status_>=200', 'checking_status_no checking', 'credit_history_all paid', 'credit_history_critical/other existing credit', 'credit_history_delayed previously', 'credit_history_existing paid', 'credit_history_no credits/all paid', 'purpose_business']


,checking_status_0<=X<200,checking_status_<0,checking_status_>=200,checking_status_no checking,credit_history_all paid,credit_history_critical/other existing credit,credit_history_delayed previously,credit_history_existing paid,credit_history_no credits/all paid,purpose_business,...,housing_own,housing_rent,job_high qualif/self emp/mgmt,job_skilled,job_unemp/unskilled non res,job_unskilled resident,own_telephone_none,own_telephone_yes,foreign_worker_no,foreign_worker_yes
0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
2,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0
3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
4,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0


In [10]:
# Join numeric columns with the encoded columns side by side
X_final = pd.concat([df[num_cols], encoded_df], axis=1)
print("Final feature table:", X_final.shape)
print("Target shape:", y.shape)

# Safety checks: no missing values, and everything is numeric
print("Missing values:", X_final.isnull().sum().sum())
print("Non-numeric columns:", X_final.select_dtypes(exclude='number').shape[1])

Final feature table: (1000, 61)
Target shape: (1000,)
Missing values: 0
Non-numeric columns: 0
